# Scientific Article Translator (Azure OpenAI)

This notebook demonstrates a small, production-minded pipeline to:

1. Fetch and parse an article from a URL
2. Clean and normalize its textual content
3. Translate the content using **Azure OpenAI**
4. Validate key functions with lightweight tests (no external calls)

> Tip: For GitHub, keep your Azure credentials in environment variables (never commit keys).

In [ ]:
# If you are running this notebook locally, install dependencies once:
# (In managed notebook environments, you may already have these.)
# !pip install -U "openai>=1.40.0" requests beautifulsoup4 python-dotenv

from __future__ import annotations

import os
import re
import textwrap
from dataclasses import dataclass
from typing import Iterable, List, Optional

import requests
from bs4 import BeautifulSoup


In [ ]:
USER_AGENT = (
    "Mozilla/5.0 (compatible; ArticleTranslator/1.0; +https://github.com/)"
)


def fetch_html(url: str, timeout_s: int = 30) -> str:
    """Fetch raw HTML from a URL.

    Args:
        url: Web page URL.
        timeout_s: Request timeout in seconds.

    Returns:
        HTML content as a string.

    Raises:
        requests.HTTPError: If the server returns an error code.
        requests.RequestException: For network-related issues.
    """
    headers = {"User-Agent": USER_AGENT}
    resp = requests.get(url, headers=headers, timeout=timeout_s)
    resp.raise_for_status()
    return resp.text


def extract_text_from_html(html: str) -> str:
    """Extract readable text from an HTML document.

    This removes script/style tags, collapses whitespace and returns a clean text
    suitable for LLM input.

    Args:
        html: Raw HTML content.

    Returns:
        Cleaned plain text.
    """
    soup = BeautifulSoup(html, "html.parser")

    for tag in soup(["script", "style", "noscript"]):
        tag.decompose()

    text = soup.get_text(separator="\n")
    text = re.sub(r"[ \t\r\f\v]+", " ", text)
    text = re.sub(r"\n{2,}", "\n\n", text)
    return text.strip()


def extract_text_from_url(url: str) -> str:
    """Convenience wrapper: fetch HTML and extract text."""
    html = fetch_html(url)
    return extract_text_from_html(html)


def chunk_text(text: str, max_chars: int = 8_000) -> List[str]:
    """Split text into chunks that are safe for LLM context windows.

    This uses a simple character-based approach to avoid extra dependencies.
    It tries to split on paragraph boundaries when possible.

    Args:
        text: Input text.
        max_chars: Maximum characters per chunk.

    Returns:
        List of chunks.
    """
    if max_chars <= 0:
        raise ValueError("max_chars must be positive")

    paragraphs = [p.strip() for p in text.split("\n\n") if p.strip()]
    chunks: List[str] = []
    current: List[str] = []
    current_len = 0

    for p in paragraphs:
        p_len = len(p) + 2  # account for separator
        if current and current_len + p_len > max_chars:
            chunks.append("\n\n".join(current).strip())
            current = [p]
            current_len = len(p)
        else:
            current.append(p)
            current_len += p_len

    if current:
        chunks.append("\n\n".join(current).strip())

    return chunks


def build_translation_prompt(target_language: str) -> str:
    """Create a stable translation instruction prompt."""
    return (
        "You are a professional translator. "
        "Translate the provided scientific/technical text to the requested language. "
        "Preserve meaning, headings, lists, equations, and citations. "
        "Do not add commentary. Do not summarize."
        f"\n\nTarget language: {target_language}"
    )


@dataclass(frozen=True)
class AzureOpenAIConfig:
    """Azure OpenAI configuration loaded from environment variables."""

    endpoint: str
    api_key: str
    api_version: str
    deployment: str

    @staticmethod
    def from_env() -> "AzureOpenAIConfig":
        endpoint = os.getenv("AZURE_OPENAI_ENDPOINT", "").strip()
        api_key = os.getenv("AZURE_OPENAI_API_KEY", "").strip()
        api_version = os.getenv("AZURE_OPENAI_API_VERSION", "2024-02-15-preview").strip()
        deployment = os.getenv("AZURE_OPENAI_DEPLOYMENT", "").strip()

        missing = [k for k, v in {
            "AZURE_OPENAI_ENDPOINT": endpoint,
            "AZURE_OPENAI_API_KEY": api_key,
            "AZURE_OPENAI_DEPLOYMENT": deployment,
        }.items() if not v]

        if missing:
            raise EnvironmentError(
                "Missing required environment variables: " + ", ".join(missing)
            )

        return AzureOpenAIConfig(
            endpoint=endpoint,
            api_key=api_key,
            api_version=api_version,
            deployment=deployment,
        )


class ArticleTranslator:
    """Translate text using Azure OpenAI (OpenAI Python SDK)."""

    def __init__(self, config: AzureOpenAIConfig):
        from openai import AzureOpenAI  # local import for clearer dependency errors

        self._client = AzureOpenAI(
            azure_endpoint=config.endpoint,
            api_key=config.api_key,
            api_version=config.api_version,
        )
        self._deployment = config.deployment

    def translate(self, text: str, target_language: str, *, max_chars: int = 8_000) -> str:
        """Translate text in chunks, concatenating the results."""
        prompt = build_translation_prompt(target_language)
        parts = chunk_text(text, max_chars=max_chars)

        outputs: List[str] = []
        for idx, part in enumerate(parts, start=1):
            response = self._client.chat.completions.create(
                model=self._deployment,
                temperature=0.2,
                messages=[
                    {"role": "system", "content": prompt},
                    {
                        "role": "user",
                        "content": (
                            f"Chunk {idx}/{len(parts)}\n\n"
                            f"{part}"
                        ),
                    },
                ],
            )
            outputs.append(response.choices[0].message.content.strip())

        return "\n\n".join(outputs).strip()


In [ ]:
# ----------------------------
# Lightweight tests (no network / no Azure calls)
# ----------------------------

def _test_extract_text_from_html() -> None:
    html = """<html><head><style>.x{}</style></head>
    <body><h1>Title</h1><script>alert(1)</script>
    <p>Hello   world</p><p>Second para.</p></body></html>"""
    text = extract_text_from_html(html)
    assert "alert" not in text
    assert "Title" in text
    assert "Hello world" in text
    assert "Second para." in text


def _test_chunk_text() -> None:
    text = "A\n\n" + ("B" * 50) + "\n\nC"
    chunks = chunk_text(text, max_chars=30)
    assert chunks, "Expected chunks"
    assert all(len(c) <= 30 or c in ["A", "C"] for c in chunks)


def _test_build_translation_prompt() -> None:
    prompt = build_translation_prompt("pt-BR")
    assert "Target language: pt-BR" in prompt
    assert "Translate" in prompt


_test_extract_text_from_html()
_test_chunk_text()
_test_build_translation_prompt()

print("All local tests passed.")


In [ ]:
# ----------------------------
# End-to-end usage (requires network + Azure OpenAI credentials)
# ----------------------------

# 1) Set env vars (recommended via .env file locally):
#    AZURE_OPENAI_ENDPOINT="https://<your-resource>.openai.azure.com"
#    AZURE_OPENAI_API_KEY="..."
#    AZURE_OPENAI_API_VERSION="2024-02-15-preview"  # or your chosen version
#    AZURE_OPENAI_DEPLOYMENT="<your-chat-deployment-name>"

# 2) Choose an article URL
url = "https://dev.to/kenakamu/azure-open-ai-in-vnet-3alo"

# 3) Fetch and extract
# NOTE: This call requires internet access in your runtime.
# text = extract_text_from_url(url)

# 4) Translate
# NOTE: This call requires valid Azure OpenAI credentials.
# config = AzureOpenAIConfig.from_env()
# translator = ArticleTranslator(config)
# translated = translator.translate(text, target_language="pt-BR")

# 5) Preview
# print(translated[:2000])
print("Notebook ready. Uncomment the lines above to run the full pipeline.")
